# 11. Character encoder

**Why.** Notebook 01 rejected the character BiGRU on cross-validation (0.813 against 0.832 in a short run). But cross-validation measures single source sentences, and notebook 09 showed that it can rank models wrongly for the competition posts: the LSTM won on CV and lost on the leaderboard. Telegram posts hold many rare street and village names that are outside the fastText vocabulary, and the EDA found that names unseen in training are recalled 24-25% of the time. A character encoder reads the letters of every word, so it can still see the shape of an unknown name (a capital letter, the ending "-ська", "вул." before it).

**What changes.** Only `char_hidden=32`: a small BiGRU over the characters of each word, whose output is joined to the word vector. Everything else is the final recipe of notebook 06 (100,000 rows per language, same folds, seed and decoding), so the comparison with notebook 06 is direct.

In [1]:
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "eda").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT), str(ROOT / "model")]
os.environ.setdefault("NER_WANDB", "off")
os.environ.setdefault("NER_DEVICE", "cpu")

import pandas as pd
from IPython.display import display

from nerkit.artifacts import combine_models, decode_split, load_logits, majority_vote, row_key, write_submission
from nerkit.config import WORK_DIR
from nerkit.data import build_cache, load_labeling_sample, load_test
from nerkit.decisions import DECODING, base_config
from nerkit.experiment import run_experiment
from nerkit.metrics import evaluate

NOTEBOOK = "11_char_encoder"
FOLDS = (0, 1, 2)
DECODE = {name: DECODING[name] for name in ("merge", "drop_empty", "trim_number", "bias")}
build_cache()
print("Runs:", {fold: base_config(fold=fold, char_hidden=32).key() for fold in FOLDS})

Runs: {0: '2e3c93d7d792', 1: '3e043599ca98', 2: '77004a4e4774'}


## Train the three folds

A finished fold is skipped on rerun. Here the three folds were trained in parallel processes with the same call, and this cell reads the finished records.

In [2]:
records = {fold: run_experiment(f"char_fold{fold}", base_config(fold=fold, char_hidden=32), NOTEBOOK) for fold in FOLDS}

char_fold0: already finished, F1 mix 0.886
char_fold1: already finished, F1 mix 0.896
char_fold2: already finished, F1 mix 0.907


## Cross-validation against notebook 06

In [3]:
saved = json.loads((ROOT / "data" / "ner_work" / "final_cv" / "summary.json").read_text(encoding="utf-8"))
before = {fold: saved["folds"][f"fold{fold}"] for fold in FOLDS}


def summary(scores, epochs, minutes):
    return dict(heldout_f1=scores["heldout"]["mix"]["f1"], precision=scores["heldout"]["mix"]["precision"],
                recall=scores["heldout"]["mix"]["recall"], posts_f1=scores["posts"]["mix"]["f1"],
                uk_unseen_recall=scores["unseen_recall"].get("uk"), labeling_f1=scores["labeling"]["pooled"]["f1"],
                epochs=epochs, minutes=minutes)


rows = []
for fold in FOLDS:
    old, new = before[fold], records[fold]
    rows.append(dict(fold=fold, model="words (nb 06)", **summary(old["scores"], old["epochs"], old["minutes"])))
    rows.append(dict(fold=fold, model="words + characters", **summary(new["scores"], len(new["history"]),
                                                                     new["total_seconds"] / 60)))
table = pd.DataFrame(rows)
display(table.set_index(["fold", "model"]).round(3))
display(table.groupby("model")[["heldout_f1", "posts_f1", "uk_unseen_recall", "labeling_f1"]].agg(["mean", "std"]).round(3))
print("parameters:", records[0]["parameters"])

heldout_f1  precision  recall  posts_f1  \
fold model                                                         
0    words (nb 06)            0.885      0.914   0.858     0.863   
     words + characters       0.886      0.895   0.876     0.869   
1    words (nb 06)            0.896      0.912   0.880     0.873   
     words + characters       0.896      0.897   0.895     0.871   
2    words (nb 06)            0.894      0.906   0.881     0.882   
     words + characters       0.907      0.921   0.893     0.880   

                         uk_unseen_recall  labeling_f1  epochs  minutes  
fold model                                                               
0    words (nb 06)                  0.600        0.606       6   39.079  
     words + characters             0.656        0.692      10   82.607  
1    words (nb 06)                  0.641        0.722      10   62.448  
     words + characters             0.678        0.651       6   49.251  
2    words (nb 06)                  0.661        0.676       6   10.514  
     words + characters             0.653        0.675      10   84.897

heldout_f1        posts_f1        uk_unseen_recall         \
                         mean    std     mean    std             mean    std   
model                                                                          
words (nb 06)           0.891  0.006    0.873  0.009            0.634  0.031   
words + characters      0.896  0.011    0.873  0.006            0.662  0.014   

                   labeling_f1         
                          mean    std  
model                                  
words (nb 06)            0.668  0.058  
words + characters       0.673  0.021

parameters: 293698


## Test predictions

The same two combinations as notebook 07: a vote (a string is kept when 2 of the 3 fold models predict it) and the average of log-probabilities and CRF transitions.

In [4]:
test_rows, labeling_rows = load_test(), load_labeling_sample()
keys = [records[fold]["key"] for fold in FOLDS]


def vote(rows, split):
    single = []
    for key in keys:
        logits, meta = load_logits(key, split)
        single.append(decode_split(logits, rows, crf=meta["crf"], **DECODE))
    return majority_vote(single, 2)


def average(rows, split):
    loaded = [load_logits(key, split) for key in keys]
    logits, crf = combine_models([l for l, _ in loaded], [m for _, m in loaded], {row_key(r): r["text"] for r in rows})
    return decode_split(logits, rows, crf=crf, **DECODE)


overview = []
for name, method, file in (("vote, 2 of 3", vote, "char_vote2.csv"), ("average", average, "char_average.csv")):
    predictions = method(test_rows, "test")
    write_submission(WORK_DIR / "submissions" / file, test_rows, predictions)
    pooled = evaluate(labeling_rows, method(labeling_rows, "labeling"))["pooled"]
    overview.append(dict(method=name, file=file, posts_with_locations=sum(bool(p) for p in predictions),
                         locations=sum(len(p) for p in predictions), labeling_f1=pooled["f1"]))
display(pd.DataFrame(overview).set_index("method").round(3))

,file,posts_with_locations,locations,labeling_f1
method,,,,
"vote, 2 of 3",char_vote2.csv,338,777,0.684
average,char_average.csv,342,803,0.684


## Leaderboard results

In [5]:
LEADERBOARD = {  # public F1, copied from Kaggle
    "words, vote 2 of 3 (notebook 07)": 0.58914,
    "words, average (notebook 07)": 0.58787,
    "words + characters, vote 2 of 3": 0.59467,
    "words + characters, average": 0.59475,
}
display(pd.Series(LEADERBOARD, name="public_f1").to_frame())

,public_f1
"words, vote 2 of 3 (notebook 07)",0.58914
"words, average (notebook 07)",0.58787
"words + characters, vote 2 of 3",0.59467
"words + characters, average",0.59475


**Conclusion.** The character encoder is the first change since notebook 07 that improves the leaderboard: 0.5947 for both combinations against 0.5891 for the word-only vote (+0.006). On cross-validation the gain is inside the noise (0.896 against 0.891), but the recall of unseen Ukrainian names rises from 0.634 to 0.662, which is the weakness the competition posts expose. So notebook 01 rejected it for the wrong reason: a short run on source sentences cannot show a gain that lives in rare names.